# 17. LSTM — all years and 5 seeds (85 models)
The walk-forward for the LSTM: for each test year 2007–2023, train **5** LSTMs (seeds 1–5) on the data up to 31 December of the year before, forecast the year, and **average** the 5 forecasts. 17 years × 5 seeds = **85 models**.

To repeat notebook 15 85 times, its steps are put into two **functions**: `train_lstm(...)` (building + training) and `forecast_lstm(...)` (testing). Nothing new happens inside them.

* **Needs:** `work/model_data.csv`. **Makes:** `work/lstm_forecasts.csv`.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/model_data.csv', index_col=0, parse_dates=True)
x = dataset[['ret', 'log_r2', 'us_ret', 'log_us_r2', 'euro_ret', 'log_euro_r2']]
y = dataset['log_rv5_fwd']
print(x.shape)
print(y.shape)

(5243, 6)
(5243,)


In [3]:
import torch
from torch import nn
torch.set_num_threads(2)


class LSTMVol(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(6, 64, num_layers=2, batch_first=True, dropout=0.2)   # 6 inputs, 64 units, 2 layers
        self.drop = nn.Dropout(0.2)
        self.head = nn.Linear(64, 1)                                                # one number out

    def forward(self, x):
        out, _ = self.lstm(x)                  # read the 22 days, one after the other
        last_day = out[:, -1]                  # the memory after the last day
        return self.head(self.drop(last_day)).squeeze(-1)

## The two functions (notebook 15 in one block)

In [4]:
from pandas import Timestamp, DateOffset, Timedelta, Series
from numpy import array, exp, random


def train_lstm(train_end, seed):
    """Notebook 15 in one function: train one LSTM on the data up to train_end."""
    # the three groups of days (notebook 14)
    train_end = Timestamp(train_end)
    val_start = train_end - DateOffset(years=1) + Timedelta(days=1)
    fit_days = y.iloc[21:].loc[:val_start - Timedelta(days=1)].dropna().index[:-5]
    val_days = y.loc[val_start:train_end].dropna().index[:-5]
    # scaling
    mean = x.loc[:fit_days[-1]].mean()
    std = x.loc[:fit_days[-1]].std()
    x_scaled = (x - mean) / std
    y_mean = y.loc[fit_days].mean()
    y_std = y.loc[fit_days].std()
    y_scaled = (y - y_mean) / y_std
    # windows
    x_fit = []
    for day in fit_days:
        position = x.index.get_loc(day)
        x_fit.append(x_scaled.values[position - 21:position + 1])
    x_val = []
    for day in val_days:
        position = x.index.get_loc(day)
        x_val.append(x_scaled.values[position - 21:position + 1])
    x_fit = torch.tensor(array(x_fit, dtype='float32'))
    y_fit = torch.tensor(array(y_scaled.loc[fit_days], dtype='float32'))
    x_val = torch.tensor(array(x_val, dtype='float32'))
    y_val = torch.tensor(array(y_scaled.loc[val_days], dtype='float32'))
    # building
    torch.manual_seed(seed)
    random.seed(seed)
    model = LSTMVol()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_function = nn.MSELoss()
    # training with early stopping
    best_val_loss = 1000000.0
    wait = 0
    for epoch in range(1, 201):
        model.train()
        order = torch.randperm(len(x_fit))
        for start in range(0, len(x_fit), 64):
            batch = order[start:start + 64]
            optimizer.zero_grad()
            loss_function(model(x_fit[batch]), y_fit[batch]).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_function(model(x_val), y_val).item()
        if val_loss < best_val_loss - 0.0001:
            best_val_loss = val_loss
            best_weights = {}
            for name, value in model.state_dict().items():
                best_weights[name] = value.clone()
            wait = 0
        else:
            wait = wait + 1
            if wait >= 15:
                break
    model.load_state_dict(best_weights)
    model.eval()
    # smearing factor on the validation days
    with torch.no_grad():
        val_pred_log = model(x_val).numpy() * y_std + y_mean
    smear = exp(y.loc[val_days].values - val_pred_log).mean()
    return {'model': model, 'mean': mean, 'std': std, 'y_mean': y_mean, 'y_std': y_std, 'smear': smear}


def forecast_lstm(m, days):
    """Testing: forecasts (variance) of a trained model for the given days."""
    x_scaled = (x - m['mean']) / m['std']
    x_days = []
    for day in days:
        position = x.index.get_loc(day)
        x_days.append(x_scaled.values[position - 21:position + 1])
    with torch.no_grad():
        pred_scaled = m['model'](torch.tensor(array(x_days, dtype='float32'))).numpy()
    return Series(exp(pred_scaled * m['y_std'] + m['y_mean']) * m['smear'], index=days)

## Test the functions: seed 1, test year 2007
This must give exactly the forecasts of notebook 15.

In [5]:
test_days = y.loc['2007-01-01':'2007-12-31'].dropna().index
m = train_lstm('2006-12-31', 1)
y_pred1 = forecast_lstm(m, test_days)
saved = read_csv(folder + 'results/lstm_forecasts.csv', index_col=0, parse_dates=True)
print('largest relative difference from the saved seed-1 forecasts:', ((y_pred1 / saved.loc[test_days, 'LSTM_seed1']) - 1).abs().max())

largest relative difference from the saved seed-1 forecasts: 4.070077608275824e-13


## All 85 models
This takes about 30 minutes on a laptop and roughly 1–1.5 hours on Colab's CPU. Set `run_all = True` to train everything here; with `run_all = False` the saved forecasts of the 85 models are loaded.

In [6]:
run_all = False

if run_all:
    from pandas import concat, DataFrame
    seed_forecasts = {1: [], 2: [], 3: [], 4: [], 5: []}
    for year in range(2007, 2024):
        test_days = y.loc[str(year) + '-01-01':str(year) + '-12-31'].dropna().index
        for seed in [1, 2, 3, 4, 5]:
            m = train_lstm(str(year - 1) + '-12-31', seed)               # building + training
            seed_forecasts[seed].append(forecast_lstm(m, test_days))     # testing
        print(year, 'done')
    lstm = DataFrame()
    for seed in [1, 2, 3, 4, 5]:
        lstm['LSTM_seed' + str(seed)] = concat(seed_forecasts[seed])
    lstm['LSTM'] = lstm.mean(axis=1)                                   # the average of the 5 seeds
else:
    lstm = read_csv(folder + 'results/lstm_forecasts.csv', index_col=0, parse_dates=True)

lstm = lstm[['LSTM', 'LSTM_seed1', 'LSTM_seed2', 'LSTM_seed3', 'LSTM_seed4', 'LSTM_seed5']]
print(len(lstm), 'forecast days')
lstm.head()

4230 forecast days


,LSTM,LSTM_seed1,LSTM_seed2,LSTM_seed3,LSTM_seed4,LSTM_seed5
2007-01-02,0.000253,0.000244,0.000250,0.000270,0.000250,0.000249
2007-01-03,0.000261,0.000249,0.000259,0.000277,0.000261,0.000258
2007-01-04,0.000273,0.000257,0.000267,0.000293,0.000282,0.000264
2007-01-05,0.000290,0.000265,0.000279,0.000315,0.000309,0.000280
2007-01-08,0.000313,0.000284,0.000296,0.000347,0.000343,0.000298


## Why average 5 seeds?
Two LSTMs trained on the same data but with different random starts give different forecasts. How different? The spread between the seeds, as a share of their average:

In [7]:
spread = lstm[['LSTM_seed1', 'LSTM_seed2', 'LSTM_seed3', 'LSTM_seed4', 'LSTM_seed5']].std(axis=1) / lstm['LSTM']
print('average spread between seeds:', round(spread.mean() * 100, 1), '%')

average spread between seeds: 13.2 %


About 13%: one seed alone would be noticeably random. The average of 5 is more stable — and in notebook 23 we ask whether the **explanations** are stable across seeds too.

In [8]:
lstm.to_csv(folder + 'work/lstm_forecasts.csv')
print('saved')

saved
